<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-agent-solution/blob/main/notebooks/EcoMarket_Agent_Solution.ipynb)

# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Proyecto Final: Agente de IA para Devoluciones

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  


## Descripción

Este notebook extiende la arquitectura RAG del Taller Práctico \#2 incorporando un **Agente de IA** capaz de ejecutar acciones autónomas sobre el sistema de EcoMarket.

A diferencia del sistema RAG anterior (puramente consultivo), este agente puede:
- **Verificar** si un pedido es elegible para devolución consultando el sistema en tiempo real.
- **Generar** una etiqueta de devolución con número de autorización (RMA) y fecha límite de envío.
- **Responder** consultas generales usando la cadena RAG del Taller 2 como herramienta adicional.

El agente implementa un patrón **Router** usando **LangGraph** `create_react_agent`: analiza la intención del usuario y decide qué herramienta invocar.

### Stack tecnológico
| Componente | Librería |
|---|---|
| Agente ReAct | `langgraph.prebuilt.create_react_agent` |
| Herramientas | `langchain_core.tools.tool` |
| LLM | `langchain_groq.ChatGroq` (llama-3.3-70b-versatile) |
| Embeddings | `intfloat/multilingual-e5-large` (HuggingFace) |
| Vector Store | ChromaDB via `langchain_chroma` |
| Interfaz | Gradio `gr.ChatInterface` |


## 1. Configuración del Entorno

In [ ]:
import warnings
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages
print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")

In [ ]:
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/requirements.txt -O requirements.txt
    !uv pip install -r requirements.txt
else:
    print('Entorno local — instala con: uv pip install -r requirements.txt')

## 2. Importaciones


In [ ]:
# Librerías estándar
import os
import uuid
import json
import random
from pathlib import Path
from datetime import datetime, timedelta

# Datos
import pandas as pd

# Entorno
if IN_COLAB:
    from google.colab import userdata
else:
    from dotenv import load_dotenv
    load_dotenv()

# LLM
from langchain_groq import ChatGroq

# Embeddings y vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

# Carga de documentos
from langchain_community.document_loaders import (
    DataFrameLoader, PyPDFLoader, JSONLoader
)

# Procesamiento de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts y cadenas (RAG — Taller 2)
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA

# Agente
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

print('Importaciones completadas correctamente.')

## 3. Carga de Datos

In [ ]:
if IN_COLAB:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(exist_ok=True)

    ![ ! -f data/FAQ.json ] && wget -q -O data/FAQ.json \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/data/FAQ.json

    ![ ! -f data/politica_devoluciones.pdf ] && wget -q -O data/politica_devoluciones.pdf \
    "https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/data/POL%C3%8DTICA%20DE%20DEVOLUCIONES.pdf"

    ![ ! -f data/pedidos_ecomarket.xlsx ] && wget -q -O data/pedidos_ecomarket.xlsx \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-agent-solution/main/data/pedidos_ecomarket.xlsx

    print('Datos descargados desde GitHub')
else:
    DATA_DIR = Path('../data')
    print(f'Datos locales en: {DATA_DIR.resolve()}')

df_pedidos = pd.read_excel(DATA_DIR / 'pedidos_ecomarket.xlsx')
print(f'Pedidos cargados: {len(df_pedidos)} registros')
print(f'Columnas: {df_pedidos.columns.tolist()}')

## 4. Inicialización del LLM, Embeddings y Vector Store


Configura el modelo **LLM** de Groq con la API key y parámetros de generación.

In [ ]:
api_key = userdata.get('GROQ_API_KEY') if IN_COLAB else os.getenv('GROQ_API_KEY')

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=api_key,
    temperature=0.3
)
print('LLM inicializado:', llm.model_name)

Inicializa el modelo de **embeddings** para representar texto como vectores.

In [ ]:
device = 'cuda' if IN_COLAB else 'cpu'

embeddings = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-large',
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)
print('Embeddings inicializados.')

Crea y configura la **vector store** para almacenar y consultar embeddings.

In [ ]:
vector_store = Chroma(
    collection_name='ecomarket_agent_collection',
    embedding_function=embeddings,
    persist_directory='./chroma_agent_db',
)
print('Vector store inicializado.')

## 5. Indexación de Documentos

In [ ]:
# Excel
df_pedidos['contenido'] = df_pedidos.apply(lambda row: ' | '.join(str(v) for v in row), axis=1)
docs_excel = DataFrameLoader(df_pedidos, page_content_column='contenido').load()

# PDF
docs_pdf = PyPDFLoader(str(DATA_DIR / 'politica_devoluciones.pdf')).load()

# JSON
docs_json = JSONLoader(
    file_path=str(DATA_DIR / 'FAQ.json'),
    jq_schema='.faq[] | "Categoría: \(.categoria)\nPregunta: \(.pregunta)\nRespuesta: \(.respuesta)"',
    text_content=True
).load()

docs = docs_excel + docs_pdf + docs_json
print(f'Documentos cargados: {len(docs)}')

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
all_splits = filter_complex_metadata(all_splits)
document_ids = vector_store.add_documents(documents=all_splits)
print(f'Sub-documentos indexados: {len(all_splits)}')

## 6. Recuperación y generación


Configura el **retriever** para buscar los 3 documentos más similares en la base vectorial.

In [ ]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

Este prompt define a EcoBot, un asistente de e-commerce que responde consultas sobre estado de pedidos y devoluciones usando información de un contexto (RAG).

Primero clasifica la intención del usuario y luego aplica un flujo específico. Incluye una regla crítica: solo usa información de pedidos si el usuario proporciona explícitamente un número de pedido, evitando errores comunes de asociación automática entre productos y pedidos.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres EcoBot, asistente de e-commerce.

PERSONALIDAD: Amable, honesto, empático y proactivo.

OBJETIVO:
Resolver consultas sobre:
1) Estado de pedidos
2) Devoluciones

REGLAS GENERALES:
- Usa SOLO el contexto proporcionado
- NO inventes información
- Si no encuentras datos, dilo claramente
- Responde SIEMPRE en español
- Tono amable, claro y profesional
- Sé conciso pero útil

--------------------------------------------------
PASO 0 — IDENTIFICAR INTENCIÓN
--------------------------------------------------
Clasifica la pregunta del usuario en UNA de estas categorías:

A) ESTADO_PEDIDO → si pregunta por envío, entrega, tracking, estado
B) DEVOLUCION → si pregunta por devolver, reembolso, cambios
C) AMBIGUO → si no está claro

- Si es AMBIGUO → pide aclaración breve antes de continuar

--------------------------------------------------
REGLA DE PRIORIDAD (CRÍTICA - OBLIGATORIA)
--------------------------------------------------

Cuando la consulta sea sobre DEVOLUCIONES:

- SOLO puedes usar un pedido si el usuario menciona explícitamente un número de pedido
  (ej: ECO-12345, pedido 123, etc.)

- PROHIBIDO:
  - Usar pedidos encontrados en el contexto si el usuario NO los mencionó
  - Inferir pedidos a partir del producto
  - Relacionar automáticamente producto ↔ pedido

- Si el usuario menciona SOLO un producto (ej: "set de cubiertos de bambú"):
  → IGNORA completamente cualquier pedido en el contexto
  → Responde como CONSULTA GENERAL (B2)

Esta regla tiene prioridad sobre cualquier otra instrucción.

--------------------------------------------------
FLUJO A — ESTADO DEL PEDIDO
--------------------------------------------------

PASO 1 — Buscar número de pedido en el contexto

PASO 2 — RESPUESTA

SI el pedido EXISTE:
- Explica el estado claramente

FORMATO OBLIGATORIO:

Hola [Nombre si está disponible, si no usa "Hola"],

Aquí tienes la información de tu pedido:

- Estado:
- Producto:
- Fecha estimada:
- Tracking:
- Última ubicación:
- Observaciones:

REGLAS ADICIONALES:
- Si está RETRASADO:
  → Empieza con disculpa sincera
  → Explica causa + nueva fecha

- Si está CANCELADO:
  → Explica estado del reembolso + plazo claro

- Si está PENDIENTE DE PAGO:
  → Indica acción requerida con tono amable

- Siempre indica el siguiente paso esperado

---

SI el pedido NO EXISTE:
- Indica que no se encontró el pedido
- Sugiere verificar el número
- Ofrece contacto:
  soporte@ecomarket.com
  900-ECO-MKT
- NUNCA inventes información

--------------------------------------------------
FLUJO B — DEVOLUCIONES
--------------------------------------------------

PASO 1 — IDENTIFICAR TIPO DE SOLICITUD

Clasifica en:

B1) DEVOLUCIÓN CON PEDIDO:
- SOLO si el usuario proporciona explícitamente un número de pedido

B2) CONSULTA GENERAL DE DEVOLUCIÓN:
- El usuario NO proporciona número de pedido
- Solo menciona un producto
- Pregunta por proceso, política o condiciones

--------------------------------------------------
CASO B1 — DEVOLUCIÓN CON PEDIDO
--------------------------------------------------

PASO 2 — Evaluar si aplica política usando el contexto

SI APLICA:

Hola [Nombre],

Claro, puedo ayudarte con la devolución de tu producto.

Sigue estos pasos:

1. [Paso 1]
2. [Paso 2]
3. [Paso 3]

- Plazo de reembolso:
- Método:

Puedes optar por crédito en tienda con 5% adicional.

---

SI NO APLICA:

Hola [Nombre],

1. Entiendo cómo te sientes con esta situación.
2. [Razón clara y humana]
3. [Alternativa concreta]

- Nunca respondas con un "no" sin alternativa

--------------------------------------------------
CASO B2 — CONSULTA GENERAL (SIN PEDIDO)
--------------------------------------------------

NO usar información de pedidos aunque exista en el contexto

FORMATO:

Hola,

Con gusto te explico cómo funciona nuestro proceso de devoluciones:

[Explicación clara basada SOLO en políticas del contexto]

Si deseas iniciar una devolución, necesitarás tu número de pedido.

¿Tienes el número de pedido? Con eso puedo ayudarte mejor.

--------------------------------------------------
CIERRE (SIEMPRE)
--------------------------------------------------

¿Puedo ayudarte con algo más?
"""),

    ("human", """Contexto:
{context}

Pregunta:
{question}
""")
])

Crea una cadena RAG que combina el modelo y el retriever para **generar** respuestas.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

## 7. Sistema de Registro de Acciones (Action Log)

Implementa la Capa 1 del sistema de monitoreo propuesto en la Fase 3. Cada invocación de una herramienta queda registrada en `logs/agent_actions.jsonl`.

In [ ]:
LOG_PATH = Path('logs')
LOG_PATH.mkdir(exist_ok=True)

SESSION_ID = str(uuid.uuid4())[:8]

def _registrar_accion(herramienta: str, inputs: dict, outputs: dict) -> None:
    entrada = {
        'timestamp': datetime.now().isoformat(),
        'session_id': SESSION_ID,
        'herramienta': herramienta,
        'input': inputs,
        'output': outputs
    }
    with open(LOG_PATH / 'agent_actions.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(entrada, ensure_ascii=False) + '\n')

print(f'Session ID: {SESSION_ID}')
print(f"Log en: {(LOG_PATH / 'agent_actions.jsonl').resolve()}")

## 8. Definición de Herramientas del Agente

El decorator `@tool` de `langchain_core.tools` convierte cada función en una herramienta que el agente ReAct puede invocar. El docstring es la descripción que el LLM lee para decidir cuándo usar cada herramienta — debe ser claro y específico.

### Herramienta 1: `verificar_elegibilidad_devolucion`

Consulta el DataFrame de pedidos en tiempo real y aplica reglas **deterministas** de elegibilidad. La decisión no depende del LLM, lo que garantiza consistencia y previene alucinaciones.

In [ ]:
@tool
def verificar_elegibilidad_devolucion(pedido_id: str) -> str:
    """Verifica si un pedido de EcoMarket es elegible para devolución.
    Úsala cuando el usuario quiera DEVOLVER un producto y proporcione un número de pedido (ej. ECO-12345).
    Retorna si el pedido es elegible y el motivo.
    Args:
        pedido_id: Identificador del pedido, por ejemplo ECO-12345.
    """
    pedido_id = pedido_id.strip().upper()
    fila = df_pedidos[df_pedidos['pedido_id'].str.upper() == pedido_id]

    if fila.empty:
        resultado = {
            'pedido_id': pedido_id,
            'elegible': False,
            'mensaje': f'No se encontró el pedido {pedido_id}. Verifica el número e intenta nuevamente.'
        }
        _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    pedido = fila.iloc[0]
    estado = pedido['estado']
    cliente = pedido['cliente']
    producto = pedido['producto']
    fecha_entrega = pedido['entrega_real']

    ESTADOS_ELEGIBLES = {'ENTREGADO', 'LISTO PARA RECOGIDA'}
    RAZONES_NO_ELEGIBLE = {
        'EN TRÁNSITO':        'el pedido aún está EN TRÁNSITO y no ha sido recibido.',
        'RETRASADO':          'el pedido está RETRASADO y aún no ha sido entregado.',
        'PROCESANDO':         'el pedido está siendo PROCESADO y no ha salido del almacén.',
        'CANCELADO':          'el pedido fue CANCELADO. Si no recibiste reembolso, contacta soporte@ecomarket.com.',
        'PENDIENTE DE PAGO':  'el pedido está PENDIENTE DE PAGO y no ha sido confirmado.',
        'RETENIDO EN ADUANA': 'el pedido está RETENIDO EN ADUANA y no está bajo control de EcoMarket.',
        'DEVUELTO':           'el pedido ya fue DEVUELTO previamente.',
    }

    if estado in ESTADOS_ELEGIBLES:
        fecha_str = str(fecha_entrega.date()) if pd.notna(fecha_entrega) else 'fecha no registrada'
        resultado = {
            'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
            'estado': estado, 'elegible': True, 'fecha_entrega': fecha_str,
            'mensaje': f'El pedido {pedido_id} ({producto}) es ELEGIBLE para devolución. Estado: {estado}. Entregado: {fecha_str}.'
        }
    else:
        razon = RAZONES_NO_ELEGIBLE.get(estado, f'el estado actual es {estado}.')
        resultado = {
            'pedido_id': pedido_id, 'cliente': cliente, 'producto': producto,
            'estado': estado, 'elegible': False,
            'mensaje': f'El pedido {pedido_id} ({producto}) NO es elegible para devolución porque {razon}'
        }

    _registrar_accion('verificar_elegibilidad_devolucion', {'pedido_id': pedido_id}, resultado)
    return json.dumps(resultado, ensure_ascii=False)

### Herramienta 2: `generar_etiqueta_devolucion`

Genera la etiqueta RMA simulada. **Verifica internamente la elegibilidad** antes de proceder: si el pedido no es elegible, la herramienta rechaza la operación independientemente de lo que el LLM haya decidido.

In [ ]:
@tool
def generar_etiqueta_devolucion(pedido_id: str) -> str:
    """Genera una etiqueta de devolución (número RMA, fecha límite, centro de envío) para un pedido elegible.
    SOLO llama esta herramienta después de confirmar con el usuario que desea proceder con la devolución.
    La herramienta verifica la elegibilidad internamente antes de generar la etiqueta.
    Args:
        pedido_id: Identificador del pedido elegible, por ejemplo ECO-12347.
    """
    pedido_id = pedido_id.strip().upper()
    fila = df_pedidos[df_pedidos['pedido_id'].str.upper() == pedido_id]

    if fila.empty:
        resultado = {'pedido_id': pedido_id, 'exito': False,
                     'mensaje': f'No se puede generar etiqueta: pedido {pedido_id} no encontrado.'}
        _registrar_accion('generar_etiqueta_devolucion', {'pedido_id': pedido_id}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    pedido = fila.iloc[0]
    estado_actual = pedido['estado']
    ESTADOS_ELEGIBLES = {'ENTREGADO', 'LISTO PARA RECOGIDA'}

    if estado_actual not in ESTADOS_ELEGIBLES:
        resultado = {'pedido_id': pedido_id, 'exito': False,
                     'mensaje': f'No se puede generar etiqueta: el pedido {pedido_id} tiene estado '
                                f'{estado_actual} y no es elegible para devolución.'}
        _registrar_accion('generar_etiqueta_devolucion', {'pedido_id': pedido_id}, resultado)
        return json.dumps(resultado, ensure_ascii=False)

    numero_autorizacion = f'RMA-{random.randint(10000000, 99999999)}'
    fecha_limite = (datetime.now() + timedelta(days=10)).strftime('%Y-%m-%d')
    CENTROS = {
        'DHL Express': 'Centro Logístico EcoMarket Norte — Av. Meridiana 350, Barcelona',
        'Correos':     'Centro Logístico EcoMarket Centro — Calle Pradillo 14, Madrid',
        'SEUR':        'Centro Logístico EcoMarket Sur — Pol. Industrial El Tablero, Sevilla',
    }
    transportista = str(pedido.get('transportista', ''))
    centro = CENTROS.get(transportista, 'Centro Logístico EcoMarket Principal — Calle Industria 42, Madrid')

    resultado = {
        'pedido_id': pedido_id, 'cliente': pedido['cliente'], 'producto': pedido['producto'],
        'exito': True, 'numero_autorizacion': numero_autorizacion,
        'fecha_limite_envio': fecha_limite, 'centro_devolucion': centro,
        'instrucciones': (
            f'1. Empaquete el producto en su embalaje original. '
            f'2. Escriba el número {numero_autorizacion} en el exterior del paquete. '
            f'3. Envíe antes del {fecha_limite} a: {centro}. '
            '4. El reembolso se procesará en 5-7 días hábiles tras recibir el producto.'
        ),
        'mensaje': f'Etiqueta de devolución generada exitosamente para el pedido {pedido_id}.'
    }
    _registrar_accion('generar_etiqueta_devolucion', {'pedido_id': pedido_id}, resultado)
    return json.dumps(resultado, ensure_ascii=False)

### Herramienta 3: `consultar_base_conocimiento`

Encapsula la cadena RAG del Taller 2. El agente la invoca para preguntas generales sobre política, FAQ o consultas de estado sin intención de devolución.

In [ ]:
@tool
def consultar_base_conocimiento(pregunta: str) -> str:
    """Responde preguntas generales sobre política de devoluciones, estado de pedidos y FAQ de EcoMarket.
    Úsala para consultas que NO requieren ejecutar una acción concreta (verificar o generar etiqueta).
    También úsala cuando el usuario pregunta por el estado de un pedido sin intención de devolverlo.
    Args:
        pregunta: La consulta del usuario en lenguaje natural.
    """
    try:
        respuesta = qa_chain.run(pregunta)
        _registrar_accion('consultar_base_conocimiento', {'pregunta': pregunta[:100]}, {'respuesta': respuesta[:200]})
        return respuesta
    except Exception as e:
        return f'Error al consultar la base de conocimiento: {str(e)}'


In [ ]:
tools = [verificar_elegibilidad_devolucion, generar_etiqueta_devolucion, consultar_base_conocimiento]
print(f"Herramientas registradas: {[t.name for t in tools]}")

## 9. Inicialización del Agente con LangChain


Los agentes combinan modelos de lenguaje con herramientas (tools) para crear sistemas que pueden razonar sobre las tareas, decidir qué herramientas usar y trabajar de forma iterativa para encontrar soluciones.

`create_agent` Proporciona una implementación de agente lista para producción.

Un agente LLM ejecuta herramientas en bucle para lograr un objetivo . El agente se ejecuta hasta que se cumple una condición de parada, es decir, cuando el modelo emite una salida final o se alcanza un límite de iteraciones.

Fuente: https://docs.langchain.com/oss/python/langchain/agents

In [ ]:
AGENT_SYSTEM_PROMPT = """
Eres EcoBot, el asistente virtual de atención al cliente de EcoMarket,
una tienda de productos sostenibles y ecológicos.

## ROL Y TONO
Actúas como un agente de soporte empático, claro y honesto. Tu tono es
cálido pero profesional: como un asesor que realmente quiere resolver el
problema, no solo cerrar el ticket. Usa frases cortas, evita tecnicismos
y adapta tu nivel de detalle a lo que el usuario necesita.

## HERRAMIENTAS Y CUÁNDO USARLAS

### `verificar_elegibilidad_devolucion`
- USA cuando: el usuario quiere devolver un producto Y ya proporcionó el
  número de pedido.
- NO USES si: el usuario no ha dado el número de pedido (pídelo primero).
- NO USES si: la consulta no involucra una devolución.

### `generar_etiqueta_devolucion`
- USA cuando: verificaste elegibilidad con resultado ELEGIBLE Y el usuario
  confirmó explícitamente que desea proceder.
- NUNCA antes de verificar elegibilidad.
- NUNCA si el usuario no confirmó.

### `consultar_base_conocimiento`
- USA cuando: preguntas generales sobre políticas, envíos, productos,
  estado de pedido (sin devolución), o cualquier duda que no sea una
  solicitud de devolución activa.

## FLUJO PARA DEVOLUCIONES (sigue este orden estrictamente)
1. Usuario menciona devolución → pregunta el número de pedido si no lo tiene.
2. Tienes el número → llama a `verificar_elegibilidad_devolucion`.
3. Resultado ELEGIBLE → informa al usuario y pide confirmación para
   generar la etiqueta.
4. Usuario confirma → llama a `generar_etiqueta_devolucion` y entrega
   instrucciones claras.
5. Resultado NO ELEGIBLE → explica la razón con empatía, sin culpar al
   usuario, y ofrece una alternativa concreta (ej: contacto con soporte,
   cambio por crédito, etc.).

## REGLAS ESTRICTAS
- Responde SIEMPRE en español.
- NUNCA inventes números de pedido, fechas, estados ni datos de contacto.
- Si una herramienta falla o devuelve un error, infórmalo con honestidad
  y sugiere que el usuario contacte soporte humano.
- No repitas información que el usuario ya te dio en el mismo turno.
- Al finalizar cada interacción resuelta, cierra con una pregunta abierta
  como: "¿Hay algo más en lo que pueda ayudarte hoy?"
""".strip()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT
)

print('Agente inicializado correctamente.')

### Función auxiliar `preguntar_al_agente`


In [ ]:
def preguntar_al_agente(consulta: str, verbose: bool = True) -> str:
    """Invoca el agente y retorna su respuesta final en texto."""
    try:
        result = agent.invoke({"messages": [HumanMessage(content=consulta)]})
        mensajes = result["messages"]

        if verbose:
            print("\n--- Pasos intermedios ---")
            for msg in mensajes:
                tipo = type(msg).__name__
                print(f"  [{tipo}]: {str(msg.content)[:200]}")
            print("--- Fin pasos intermedios ---\n")

        # Buscar el AIMessage con contenido más rico (no el cierre vacío)
        respuesta_final = ""
        for msg in reversed(mensajes):
            if hasattr(msg, "content") and len(str(msg.content).strip()) > 30:
                respuesta_final = str(msg.content)
                break

        # Si no encontró nada útil, tomar el último mensaje
        if not respuesta_final:
            respuesta_final = str(mensajes[-1].content)

        return respuesta_final

    except Exception as e:
        return f"Error del agente: {str(e)}"

## 10. Evaluación del Comportamiento del Agente

Se prueban 7 escenarios para verificar que el agente:
1. Invoca las herramientas correctas según la intención.
2. Maneja pedidos elegibles (flujo completo) e inelegibles (respuesta empática).
3. Responde consultas generales usando solo el RAG.
4. Maneja pedidos inexistentes y consultas fuera de dominio.

### Prueba 1 — Devolución con pedido ELEGIBLE (flujo completo: verificar → generar etiqueta)

In [ ]:
respuesta = preguntar_al_agente(
    'Quiero devolver mi pedido ECO-12347, la botella llegó con un golpe. '
    'Si aplica, por favor genera la etiqueta de devolución.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Prueba 2 — Devolución con pedido NO ELEGIBLE (estado: EN TRÁNSITO)

In [ ]:
respuesta = preguntar_al_agente(
    'Quiero devolver mi pedido ECO-12345, ya no lo necesito.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Prueba 3 — Devolución con pedido CANCELADO

In [ ]:
respuesta = preguntar_al_agente(
    'Necesito hacer la devolución del pedido ECO-12349.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Prueba 4 — Consulta general de política (sin número de pedido)

In [ ]:
respuesta = preguntar_al_agente(
    '¿Cuántos días tengo para hacer una devolución? ¿Qué productos no aplican?'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Prueba 5 — Consulta de estado de pedido (sin intención de devolver)

In [ ]:
respuesta = preguntar_al_agente(
    '¿Cuál es el estado de mi pedido ECO-12346? ¿Cuándo llegará?'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Prueba 6 — Pedido INEXISTENTE

In [ ]:
respuesta = preguntar_al_agente(
    'Quiero devolver el pedido ECO-99999.'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Prueba 7 — Consulta fuera de dominio

In [ ]:
respuesta = preguntar_al_agente(
    '¿Me puedes recomendar una receta de pasta carbonara?'
)
print('\n' + '='*60)
print('RESPUESTA FINAL:')
print('='*60)
print(respuesta)

### Revisión del Log de Acciones

In [ ]:
log_file = LOG_PATH / 'agent_actions.jsonl'
if log_file.exists():
    with open(log_file, encoding='utf-8') as f:
        lineas = f.readlines()
    print(f'Total de acciones registradas: {len(lineas)}\n')
    for linea in lineas[-5:]:
        entrada = json.loads(linea)
        print(f"  [{entrada['timestamp']}] {entrada['herramienta']} | input: {entrada['input']}")
else:
    print('No hay acciones registradas aún.')

## Fase 4: Despliegue con Gradio

### Justificación de Gradio sobre Streamlit

- `gr.ChatInterface` provee interfaz de chat lista en menos de 10 líneas, ideal para la sustentación.
- Streamlit requiere gestionar `st.session_state` manualmente para el historial.
- `share=True` despliega en Hugging Face Spaces sin configuración adicional.
- La interfaz tipo chat es más natural para demostrar el comportamiento del agente con distintos prompts.

In [ ]:
try:
    import gradio as gr
    print(f'Gradio disponible: v{gr.__version__}')
except ImportError:
    !pip install gradio -q
    import gradio as gr
    print(f'Gradio instalado: v{gr.__version__}')

In [ ]:
import gradio as gr

def responder(mensaje: str, historial: list):
    """Conecta Gradio con el agente."""
    if not mensaje.strip():
        yield "Por favor, escribe tu consulta para que pueda ayudarte."
        return
    respuesta = preguntar_al_agente(mensaje, verbose=False)
    yield respuesta

demo = gr.ChatInterface(
    fn=responder,
    title="🌱 EcoBot — Asistente de EcoMarket",
    description=(
        "Bienvenido al asistente inteligente de EcoMarket. Puedo ayudarte con:\n"
        "• Estado de tus pedidos\n"
        "• Proceso de devoluciones\n"
        "• Preguntas sobre nuestra política de devoluciones\n\n"
        "Ejemplos: *¿Cuál es el estado del pedido ECO-12345?* · "
        "*Quiero devolver el pedido ECO-12347* · *¿Cuántos días tengo para devolver?*"
    ),
    examples=[
        "¿Cuál es el estado de mi pedido ECO-12346?",
        "Quiero devolver el pedido ECO-12347, llegó dañado.",
        "¿Cuántos días tengo para hacer una devolución?",
        "Necesito devolver el pedido ECO-12345.",
        "¿Qué productos no se pueden devolver?"
    ],
    theme=gr.themes.Soft(primary_hue="green"),
    editable=False,
)

demo.launch(share=IN_COLAB)